In [232]:
import torch 
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

import math

torch.manual_seed(1337);

In [14]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
len(text)

1115394

In [8]:
chars = sorted(list(set(text)))
print(''.join(chars))
vocab_size = len(chars)
vocab_size


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


65

In [13]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] 
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [24]:
data = torch.tensor(encode(text), dtype=torch.long)
data.shape, data.dtype

(torch.Size([1115394]), torch.int64)

In [26]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [29]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [30]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [31]:
y = train_data[1:block_size+1]
y

tensor([47, 56, 57, 58,  1, 15, 47, 58])

In [35]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [45]:
xb, yb

(tensor([[24, 43, 58,  5, 57,  1, 46, 43],
         [44, 53, 56,  1, 58, 46, 39, 58],
         [52, 58,  1, 58, 46, 39, 58,  1],
         [25, 17, 27, 10,  0, 21,  1, 54]]),
 tensor([[43, 58,  5, 57,  1, 46, 43, 39],
         [53, 56,  1, 58, 46, 39, 58,  1],
         [58,  1, 58, 46, 39, 58,  1, 46],
         [17, 27, 10,  0, 21,  1, 54, 39]]))

In [66]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    
    def forward(self, idx, targets=None):
        # idx (B, T)
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    
    def generate(self, idx, max_new_tokens):
        # idx (B, T)
        for _ in range(max_new_tokens):
            logits, _loss = self(idx)
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
logits.shape, loss

(torch.Size([32, 65]), tensor(4.5564, grad_fn=<NllLossBackward0>))

In [67]:
-math.log(1/vocab_size)

4.174387269895637

In [76]:
decode([1])

' '

In [85]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [102]:
batch_size = 32
for steps in range(10000):
    xb,  yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.457803726196289


In [107]:
idx = torch.zeros((1, 1), dtype=torch.long) # (B, C)
gen = m.generate(idx, max_new_tokens=300)[0].tolist()
print(decode(gen))


Fis un t ro be,

We,
Wee heth no bemy u, manfou Hefoulthat.
Aned oslus,
Lawind osorand ccumu ken
Tousede? h wnenofoof tanous s thin acestiory t ad RE:
BOFINIUSo angowailepeno ngorrveoowe f I bs The;
ANThubesang vearomed 'd, tumotheen any ol ce?
WAs basare the hee d OULAfis, stheeweve ou se pld or m 


# Attention

In [ ]:
# APPROACH 1 (NAIVE ITER)

In [110]:
torch.manual_seed(1337);

In [114]:
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [117]:
xbow = torch.zeros((B,T,C))

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t+1, C)
        xbow[b,t] = torch.mean(xprev, 0)

In [119]:
x[0], xbow[0]

(tensor([[-0.8345,  0.5978],
         [-0.0514, -0.0646],
         [-0.4970,  0.4658],
         [-0.2573, -1.0673],
         [ 2.0089, -0.5370],
         [ 0.2228,  0.6971],
         [-1.4267,  0.9059],
         [ 0.1446,  0.2280]]),
 tensor([[-0.8345,  0.5978],
         [-0.4429,  0.2666],
         [-0.4610,  0.3330],
         [-0.4100, -0.0171],
         [ 0.0738, -0.1210],
         [ 0.0986,  0.0153],
         [-0.1193,  0.1425],
         [-0.0863,  0.1532]]))

In [ ]:
# APPROACH 2 (MATRIX FORM)

In [136]:
a = torch.tril(torch.ones(3, 3))
a /= a.sum(dim=1, keepdim=True)
a, a.shape

(tensor([[1.0000, 0.0000, 0.0000],
         [0.5000, 0.5000, 0.0000],
         [0.3333, 0.3333, 0.3333]]),
 torch.Size([3, 3]))

In [137]:
b = torch.randint(0, 10, (3,2)).float()
b, b.shape

(tensor([[1., 4.],
         [9., 2.],
         [5., 0.]]),
 torch.Size([3, 2]))

In [138]:
a @ b

tensor([[1., 4.],
        [5., 3.],
        [5., 2.]])

In [143]:
wei = torch.tril(torch.ones(T,T))
wei /= wei.sum(dim=1, keepdim=True)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [144]:
wei.shape, x.shape

(torch.Size([8, 8]), torch.Size([4, 8, 2]))

In [146]:
x[0]

tensor([[-0.8345,  0.5978],
        [-0.0514, -0.0646],
        [-0.4970,  0.4658],
        [-0.2573, -1.0673],
        [ 2.0089, -0.5370],
        [ 0.2228,  0.6971],
        [-1.4267,  0.9059],
        [ 0.1446,  0.2280]])

In [145]:
# (T,T) @ (B,T,C) -> (B,T,C)
# (B,T,T) @ (B,T,C) (broadcast) 

# for each batch, (T,T) @ (T,C) -> (T,C)

In [150]:
xbow2 = wei@x
torch.allclose(xbow2, xbow)

True

In [ ]:
# APPROACH 3 (GENERAL SOFTMAX)

In [162]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
print(wei)
wei = F.softmax(wei, dim=1)
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [169]:
xbow3 = wei@x
torch.allclose(xbow3, xbow)
# this generalizes trailing averages, but is more general bc we can have now weighted trailing averages, and learn them

True

In [171]:
torch.arange(7)

tensor([0, 1, 2, 3, 4, 5, 6])

# Self-Attention

In [179]:
B,T,C = 4,8,32
x = torch.randn(B,T,C)

tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x

x.shape, out.shape

(torch.Size([4, 8, 32]), torch.Size([4, 8, 32]))

In [180]:
# the idea is to build wei in a sensible way (using q, k)
# a token will have:
# query -> "what it wants to know"
# key -> "what info it contains"
# dot product of this gives a sensible way to construct affinities (see how q, and k are aligned)
# this is very geometric!

In [236]:
# one head of self-attention

B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)

k = key(x) # (B,T,16)  
q = query(x) # (B,T,16)

# ^ so far k,q are created in parallel independently, no communication
# now we mix them with dot product to crate the affinities:

wei = q @ k.transpose(-2, -1) # (B,T,16) @ (B,16,T) -> (B,T,T)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1) # (B,T,T)

out = wei @ x

print(wei.shape, x.shape)

torch.Size([4, 8, 8]) torch.Size([4, 8, 32])


In [237]:
# interpretation:

a = wei[0,5] # token 5 affinities
b = x[0] # token channels (T,C)
explicit = a @ x[0]
c = out[0,5] # token 5 weighted channels by affinities/atention

print(explicit.shape)
print(c.shape)

print(torch.allclose(explicit, c))

print(a.shape, x.shape, wei.shape)

torch.Size([32])
torch.Size([32])
True
torch.Size([8]) torch.Size([4, 8, 32]) torch.Size([4, 8, 8])


In [190]:
x.shape, key.weight.T.shape, k.shape, q.shape

(torch.Size([4, 8, 32]),
 torch.Size([32, 16]),
 torch.Size([4, 8, 16]),
 torch.Size([4, 8, 16]))

In [240]:
# how the dot product works:

print(k.shape)
kt = k.transpose(-2,-1)
print(kt.shape)

print(q.shape, kt.shape)
wei = q @ k.transpose(-2, -1)
print(wei.shape)

c = wei[0,3,1]
a = q[0,3,:] # what token 3 wants
b = k[0,1,:] # what token 1 has
aff = a @ b
a.shape, b.shape, c.shape, aff.shape

c, aff

torch.Size([4, 8, 16])
torch.Size([4, 16, 8])
torch.Size([4, 8, 16]) torch.Size([4, 16, 8])
torch.Size([4, 8, 8])


(tensor(-0.8014, grad_fn=<SelectBackward0>),
 tensor(-0.8014, grad_fn=<DotBackward0>))

In [ ]:
# not symmetric in genreal
# W = QKt != KQt in general

In [246]:
# the real thing:
# one head of self-attention

B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

# think of x as containing private information of token i at row i
# then we transform that information in k,q,v
# row i of k is -> info of what info token i contains
# row i of q is -> info of what info token i is looking for
# row i of v is -> raw info of token i

k = key(x) # (B,T,16)  
q = query(x) # (B,T,16)
v = value(x) # (B,T,16)

# ^ so far k,q and v are created in parallel independently, no communication
# now we mix k,q with dot product to crate the affinities:

wei = q @ k.transpose(-2, -1) # (B,T,16) @ (B,16,T) -> (B,T,T)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1) # (B,T,T)

# given batch out row i is what token i obtains (information weighted by what he wants, what is available in context, and the raw information in context)
out = wei @ v # (B,T,T) @ (B,T,16) -> (B,T,16)

out.shape

torch.Size([4, 8, 16])

## Scaled self-attention (numerical bahaviour)

In [288]:
k = torch.randn(B,T,head_size) # 0 mean, 1 std
q = torch.randn(B,T,head_size) # 0 mean, 1 std
wei = q @ k.transpose(-2,-1) # * head_size**-0.5

In [289]:
k.std(), k.mean(), q.std(), q.mean(), wei.std(), wei.mean(), head_size**0.5

(tensor(0.9881),
 tensor(0.0185),
 tensor(0.9872),
 tensor(0.0439),
 tensor(4.0056),
 tensor(-0.0393),
 4.0)

In [291]:
k = torch.randn(B,T,head_size) # 0 mean, 1 std
q = torch.randn(B,T,head_size) # 0 mean, 1 std
wei = q @ k.transpose(-2,-1) * head_size**-0.5 # normalize to 1std

wei.std()

# softmax, same vector but scalar scaled behaves like a max, you only attend to one token, bad, especially at init

tensor(1.0805)